# 05 — Determinant selection: the oracle bound, CIPSI, and bond breaking

**What this notebook answers:** *before* claiming a quantum sampler beats a
classical one, how do you check the test system can even show a difference —
and where does that difference actually appear?

This is the study that produced the Section V results in the thesis and the
talk. It is the last notebook chronologically and the one whose numbers get
quoted most, so it carries the most provenance detail.

## Read this first if you are picking the project back up

Three things live here that exist nowhere else in the pipeline:

| quantity | what it is | code |
|---|---|---|
| **w₁** (dominant weight) | \|c₀\|² of the largest CI amplitude. The correlation dial: ≈0.9 single-reference, <0.5 strongly correlated | `det_analysis.weight_curve()` |
| **oracle bound** | best energy obtainable from *any* N determinants, by exact amplitude ranking. A ceiling, not a method | `det_analysis.oracle_curve()` |
| **CIPSI** | classical selected CI (1973). Same Hamiltonian, same budget, zero quantum input | `det_expansion.cipsi_from_scratch()` |

None of them are part of `quenais-run`. They are diagnostics that answer a
question the pipeline itself cannot: *given a fixed determinant budget, how
much of the remaining error is the sampler's fault?*

**Most of this notebook runs with no PySCF and no GPU** — it reads stored
results from `tests/regression/golden/`. Cells that need a live PySCF
environment are marked **[NEEDS PYSCF]** and can be skipped on a laptop.

In [ ]:
import json
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO = Path("..")
GOLDEN = REPO / "tests" / "regression" / "golden"

def read_csv(path):
    with open(path) as fh:
        return list(csv.DictReader(fh))

# What has already been computed and committed:
for system in ["LiH", "N2", "ScH"]:
    have = sorted(p.name for p in (GOLDEN / system).glob("stage*"))
    print(f"{system:4}  {have if have else '(no stage0/stage1 results stored)'}")

Note N₂ has no stored `stage0`/`stage1` files. That is deliberate, and it is
the first gotcha in this notebook — see the next section.

---

## 1. The budget guard — why N₂'s golden active space is useless here

`tests/regression/golden/N2/` uses the **(4e,4o)** active space, which is
36 determinants total. Any sensible selection budget (we use 200) is *larger
than the entire space*, so CIPSI and the oracle both get handed everything and
both come out exact. The comparison is vacuous — it measures nothing.

This actually happened. An early version of `run_correlation_scan.py` used a
4-orbital space with a 200-determinant budget, every method returned an
identical number, and the plot looked like a perfectly valid comparison.
See `docs/reproducibility.md` §1.

The fix is a hard guard, not a warning:

```python
if budget >= 0.5 * space.ndet:
    raise ValueError(...)   # "Every method would look identical."
```

**Consequence for anyone re-running this:** the correlation scan uses the
**full valence (10e,8o)** space — MOs 2–9, i.e. 3136 determinants — *not* the
golden (4e,4o). Those are two different experiments:

| space | determinants | what it is for |
|---|---|---|
| (4e,4o), MOs 5–8 | 36 | reproducing the golden regression numbers |
| (10e,8o), MOs 2–9 | 3136 | measuring selection quality |

The (10e,8o) space is also the physically right one: breaking a triple bond
needs all three bonding/antibonding pairs, or the stretched geometries are not
actually strongly correlated.

---

## 2. ScH — reading the stored result

Start here because ScH is the system the project spent the most time on, and
the stored files answer the question directly.

`stage0_ScH_summary.json` is written by `tools/run_stage0.py`. Its fields are
the whole Section V argument in one object.

> **Every ScH number below rests on a hand-forced active space.**
> ASF under-selects for 3d elements — on ScH it kept 4 orbitals but only 2
> active electrons. All stored ScH results use
> `force_active_space=[9, 10, 11, 12, 13, 14]`, i.e. `CAS(4e,6o)`, 22 qubits.
> `reference_values.py` records this as `"force_active_space": [9, ..., 14]`
> (LiH and N₂ are `None` — automatic).
>
> This matters here specifically: **w₁ = 0.895 is a property of that chosen
> space, not of ScH in the abstract.** A larger active space would admit more
> configurations and could lower w₁. So "ScH cannot discriminate selection
> methods" is a statement about ScH *in CAS(4e,6o)* — which is the space every
> solver in this project was benchmarked in, so the conclusion stands for the
> benchmarks, but it is not a claim about scandium hydride as a molecule.
>
> Setting it: notebook 04 §"Forcing the active space", or
> `--force-active-space 9 10 11 12 13 14`.

In [ ]:
summary = json.loads((GOLDEN / "ScH" / "stage0_ScH_summary.json").read_text())
for k, v in summary.items():
    print(f"{k:22} {v}")

Three fields matter:

- **`weight_targets["0.9"] = 2`** — two determinants out of 108,900 capture 90%
  of the wavefunction. Read `stage0_ScH_curve.csv` row 1 for the exact w₁.
- **`n_chem = 144`** — only 144 determinants (0.13% of the space) are needed to
  reach chemical accuracy, *if you pick the right ones*.
- **`headroom_mha = 20.895`** — the gap between the oracle at the tested budget
  and what GQE actually achieved. This is the number quoted as "GQE 20.9 mHa"
  in the talk.

In [ ]:
curve = read_csv(GOLDEN / "ScH" / "stage0_ScH_curve.csv")
w1 = float(curve[0]["weight_captured"])
print(f"w1 (ScH, r=1.78 A, forced CAS(4e,6o)) = {w1:.4f}")
print()
print(f"  {'N':>7}  {'% of space':>10}  {'weight':>10}  {'err (mHa)':>10}")
for row in curve:
    n = int(row["n_det"])
    if n in (1, 2, 20, 104, 144, 373, 1023, 2439):
        print(f"  {n:>7}  {100*float(row['fraction']):>9.3f}%  "
              f"{float(row['weight_captured']):>10.6f}  "
              f"{float(row['err_projected_mha']):>10.4f}")

w₁ = 0.895. **This is a single-reference system by the correlation measure** —
one configuration carries 89.5% of the state — despite being a transition-metal
system, which is exactly the class you would expect to be strongly correlated.

That is the trap. ScH *looks* like a hard problem (22 qubits, 108,900
determinants, a 3d shell) and is *not* hard in the way that distinguishes
selection methods.

Now the same budget, classical vs quantum:

In [ ]:
cipsi = read_csv(GOLDEN / "ScH" / "stage1_ScH_cipsi.csv")
print(f"  {'N':>6}  {'CIPSI err':>12}  {'oracle err':>12}  {'gap to oracle':>14}")
for r in cipsi:
    print(f"  {int(r['n_det']):>6}  {float(r['err_cipsi_mha']):>10.4f} mHa  "
          f"{float(r['err_oracle_mha']):>10.4f} mHa  "
          f"{float(r['gap_to_oracle_mha']):>+12.4f} mHa")

best = cipsi[-1]
e_cipsi = float(best["err_cipsi_mha"])
e_gqe = summary["headroom_mha"]
print(f"\n  CIPSI at N={best['n_det']}:  {e_cipsi:.4f} mHa")
print(f"  GQE   at same budget:  {e_gqe:.4f} mHa")
print(f"  ratio: {e_gqe / e_cipsi:,.0f}x in favour of the 1973 classical method")

A 1973 classical algorithm beats the quantum pipeline by ~4,900× on this
system.

**Do not read that as "GQE is bad."** Read the `gap_to_oracle_mha` column:
CIPSI is within 0.0002 mHa of the *theoretical best possible* answer at that
budget. There is no room left for any method — quantum or classical — to show
an advantage. ScH cannot discriminate selection methods, so it cannot be used
to evaluate them.

This is the finding that reframes the project's whole solver history: SQD →
SKQD → SqDRIFT → GQE were each evaluated primarily on ScH, where no method
could have shown a measurable difference.

---

## 3. Bond breaking — the experiment that *can* discriminate

If ScH is stuck at w₁ = 0.895, you need a system where w₁ can be **tuned**.
Stretching a bond does exactly that: near equilibrium one configuration
dominates; near dissociation the wavefunction spreads over many near-degenerate
configurations and there is nothing for a perturbative criterion to anchor on.

`tools/run_correlation_scan.py` holds the active space **fixed** across the scan
so geometry is the only variable — otherwise ASF would pick a different space at
each bond length and you would be comparing different embeddings, not different
correlation strengths.

### Running it

```bash
python tools/run_correlation_scan.py                          # N2, default 8 geometries
python tools/run_correlation_scan.py --distances 1.0977 1.5 1.8 2.1 2.4
python tools/run_correlation_scan.py --molecule Cr2 --threads 24
```

Writes `scans/<mol>/correlation_scan_<mol>.csv` and `.json`.

### It does NOT use DMET, on purpose

The first version ran the full DMET pipeline at each geometry and *every point*
failed the embedded-SCF check by 0.3–0.7 Ha — equilibrium included. Cause:
N₂/STO-3G has only 10 orbitals, so an 8-orbital impurity plus bath spans the
whole molecule, leaving DMET no environment to fold into `e_core`. The core
potential and electron count then double-count and the embedding Hamiltonian is
meaningless.

The question being asked here is about a *Hamiltonian*, not an embedding, so the
scan builds the CAS Hamiltonian directly with PySCF and skips DMET entirely.
See `DEV_NOTES.md` §4 and `docs/reproducibility.md` §2.

### The measured scan

These are the committed results from the run in `scan_A.txt`
(N₂/STO-3G, forced (10e,8o) MOs 2–9, budget 200 of 3136 determinants = 6.4%).
Re-running the same command should reproduce them; if it does not, jump to the
fingerprints section below before doing anything else.

In [ ]:
# N2/sto-3g, (10e,8o), budget=200. Source: scan_A.txt (tools/run_correlation_scan.py)
scan = [
    # r,      w1,      CIPSI,   oracle,  gap,     N_chem
    (1.0977, 0.9173,  0.1182,  0.1200, -0.0018, 134),
    (1.3000, 0.8523,  0.2981,  0.2913, +0.0069, 134),
    (1.5000, 0.7373,  0.4646,  0.3743, +0.0903, 191),
    (1.8000, 0.4257,  0.5186,  0.3968, +0.1218, 191),
    (2.1000, 0.1916,  1.6396,  0.1101, +1.5295, 134),
    (2.4000, 0.1112,  0.9842,  0.0405, +0.9438,  95),
    (2.8000, 0.0782,  0.2394,  0.0091, +0.2303,  67),
    (3.2000, 0.0677,  0.0382,  0.0060, +0.0322,  23),
]

def verdict(w1_, gap):
    # Two regimes, and they are NOT the same cut. Above w1~0.74 classical
    # selection is effectively optimal. Below it a gap opens -- but it closes
    # again once the bond is fully broken, which is the counterintuitive part.
    if w1_ > 0.74:
        return "classical optimal (nothing to win)"
    if gap > 0.5:
        return "<<< HEADROOM for a better sampler"
    return "past threshold, but gap closing again"

print(f"  {'r (A)':>7} {'w1':>7} {'CIPSI':>9} {'oracle':>9} {'gap':>9}  verdict")
for r, w1_, c, o, g, _ in scan:
    print(f"  {r:>7.4f} {w1_:>7.4f} {c:>7.4f}   {o:>7.4f}   {g:>+7.4f}   "
          f"{verdict(w1_, g)}")

In [ ]:
rs   = [s[0] for s in scan]
w1s  = [s[1] for s in scan]
gaps = [s[4] for s in scan]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(rs, w1s, "o-", color="#17828F")
ax1.axhline(0.74, ls="--", color="#C00000", lw=1)
ax1.text(2.4, 0.77, "w1 = 0.74 threshold", color="#C00000", fontsize=9)
ax1.set_xlabel("N-N bond length (A)"); ax1.set_ylabel("dominant weight w1")
ax1.set_title("Stretching tunes correlation")

ax2.plot(w1s, gaps, "o-", color="#C55A11")
ax2.axvline(0.74, ls="--", color="#C00000", lw=1)
ax2.axhline(0, color="0.7", lw=0.8)
ax2.set_xlabel("dominant weight w1"); ax2.set_ylabel("CIPSI - oracle (mHa)")
ax2.set_title("Where classical selection starts to lose")
ax2.invert_xaxis()

fig.tight_layout()

**The threshold: w₁ ≈ 0.74.** Above it the gap is ~0 — classical perturbative
selection is effectively optimal, and nothing can beat it. Below it the gap
opens up.

**The peak is at 2.1 Å, not at full dissociation.** This is the least intuitive
result in the whole study and worth stating explicitly to anyone who asks: at
2.8–3.2 Å the gap *shrinks again*. Once the bond is fully broken the wavefunction
becomes a clean, near-degenerate combination that CIPSI handles fine (and
`N_chem` drops to 23). The hard regime is **intermediate stretching**, where the
state is genuinely multiconfigurational but has no simple structure.

So the interesting geometry is 2.1 Å — w₁ = 0.19, CIPSI 1.64 mHa vs an oracle of
0.11 mHa. That is a 15× headroom for a better selection method to exploit, and
it is the only geometry in this scan where GQE has actually been tested.

---

## 4. The 2.1 Å result

At that one geometry, with the same 200-determinant budget:

| method | error vs exact CASCI | notes |
|---|---|---|
| oracle bound | 0.110 mHa | theoretical ceiling — not achievable by any real method |
| **DMET + GQE** | **0.153 mHa** | this pipeline |
| CIPSI (1973) | 1.640 mHa | classical baseline, identical Hamiltonian and budget |

GQE beats the classical baseline **10×** at equal cost, and lands 0.04 mHa from
the theoretical best-possible answer.

### Caveats to state out loud before quoting this

1. **One geometry.** Of the eight scanned, this is the only one past the
   threshold where GQE has been run. It is one point, not a curve.
2. **Seed count.** Check how many independent `--gqe-seed` values back this
   number before quoting a spread. See the warning in `docs/reproducibility.md`
   §5 — a script that launches "three repeats" without varying `--gqe-seed`
   reproduces failure mode 5 exactly, and that mistake cost hours once already.
3. **The oracle is pessimistic.** Top-N by amplitude is not the energy-optimal
   N-determinant set, so E(top-N) ≥ E(optimal-N). The measured headroom is a
   conservative bound — the true headroom is at least what is reported.

In [ ]:
# [NEEDS PYSCF] Reproduce the 2.1 A comparison from scratch.
#
#   python tools/run_correlation_scan.py --distances 2.1
#
# Or, in-process, against a step-2 pickle for that geometry:
#
# from quenais.quantum.gqe_adapter import load_from_dmet_pickle
# from quenais.quantum import det_analysis as da, det_expansion as dx
#
# mol = load_from_dmet_pickle("scans/N2/N2_r2.1000/results/step2_hamiltonian.pkl")
# e_exact, flat, space = da.casci_vector(mol)
# order, cum = da.weight_curve(flat)
# print("w1 =", cum[0])
#
# e_oracle = da.projected_energy(mol, order[:200], space=space)
# sel, history = dx.cipsi_from_scratch(mol, 200, space=space)
# e_cipsi = history[-1]["energy"]
# print(f"oracle {1e3*(e_oracle-e_exact):.4f} mHa   CIPSI {1e3*(e_cipsi-e_exact):.4f} mHa")
print("See the commented recipe above — needs a PySCF environment.")

---

## 5. The dissociation curve — the picture behind all of this

`tools/run_dissociation.py` produces the standard strong-correlation figure:
every classical method against exact CASCI, across bond length.

```bash
python tools/run_dissociation.py                       # N2, 31 geometries, sto-3g
python tools/run_dissociation.py --molecule N2 --distances 1.0977 1.5 2.1 3.0
python tools/run_dissociation.py --ncas 8 --nelecas 10 --out figs/
```

Outputs `dissociation_<mol>.csv` and `dissociation_<mol>.pdf` (two panels:
absolute energy, and error vs exact).

### What to look for

As the bond breaks, **CCSD does not merely lose accuracy — it crosses through
the exact answer and ends up below it.** Coupled cluster is non-variational, so
there is no bound stopping it, and its amplitudes diverge: measured
max\|t₂\| = 0.65 at 1.8 Å and **0.84 at 2.1 Å**, where a healthy value is below
0.1.

That crossing is the cleanest available demonstration that a system is strongly
correlated, and it is the direct motivation for everything downstream — it is
precisely where a method treating many configurations on equal footing becomes
necessary.

### Comparability caveat (put this in the figure caption)

HF/MP2/CCSD/CCSD(T) are **all-electron**; CASCI is within the active space with
a **frozen core**. For N₂/STO-3G with (10e,8o) the frozen 1s pair contributes
almost no correlation, so the comparison is meaningful — but these are not
identical theory levels and the caption should say so.

---

## 6. Gotchas — read before trusting a re-run

Five mechanisms in this project have each produced a smooth, plausible, **wrong**
result, and three were invisible in the energy alone. Full writeups in
`docs/reproducibility.md`; these are the ones specific to *this* study.

### 6.1 The fingerprints tell you which layer broke

`measure()` records `civec_fp` and `order_fp` at every geometry. If a re-run
disagrees, diff those first — they localise the problem instead of leaving it to
guesswork:

| `civec_fp` | `order_fp` | energies | meaning |
|---|---|---|---|
| differ | — | — | the exact CI vector itself is not reproducible |
| same | differ | — | amplitudes near the cutoff are tied; ranking is arbitrary, state is fine |
| same | same | differ | the instability is downstream, in the subspace diagonalisation |

### 6.2 ARPACK's random start vector

`scipy.sparse.linalg.eigsh` seeds from a *random* start vector when `v0` is not
given. Two runs of an identical calculation then converge along different Krylov
paths and return eigenvectors that agree on the eigenvalue but reorder near-equal
amplitudes — an irreproducible determinant *ranking* from a perfectly correct
energy. `v0` is now mandatory in `projected_energy`, fixed to
`np.full(n, 1/sqrt(n))`. **Do not remove it.**

### 6.3 Degenerate orbitals fixed only up to a rotation

Without `symmetry=True` on the PySCF molecule, RHF fixes a degenerate π pair only
up to an arbitrary rotation within that subspace. Two runs converge to two
different valid orbital bases: **total energies matched to 14 decimal places, CI
vectors did not.** This is the sharpest example in the project of an
energy-only check being worthless.

### 6.4 Tied determinant groups at the budget cutoff

Symmetry-equivalent determinants carry exactly equal weight. Taking some of a
tied group and not the rest breaks the trial space's symmetry and raises the
energy for a reason unrelated to selection quality. `measure()` extends the cut
to whole tied groups — that is why you see `budget=200 (tied group -> using 201
dets)` in the scan output. The `cutoff_ratio` column reports how tied the cut was.

### 6.5 Every oracle point must be variational

`oracle_curve()` raises immediately if a projected energy falls below the exact
reference:

```
Projected energy fell BELOW the exact reference at N=...
This is not possible variationally -- the determinant indexing or the
reference is wrong. Stop and fix before reading any other number.
```

If you see this, the determinant indexing is wrong (most likely alpha/beta
divisor — see `docs/limitations.md`) and **everything after it is noise**. Do not
work around it.

---

## 7. Re-run recipes

Copy-paste, from the repo root with `quenais-env` active.

```bash
# Stage 0 — weight curves + oracle bound (LiH seconds, ScH minutes)
python tools/run_stage0.py --system ScH --threads 24

# Stage 1 — CIPSI at matched subspace sizes
python tools/run_stage1.py --system ScH --threads 24

# The correlation scan (the bond-breaking study)
python tools/run_correlation_scan.py --molecule N2 --threads 24

# The dissociation curve figure
python tools/run_dissociation.py --molecule N2 --out figs/

# GQE at one geometry, with an EXPLICIT seed — see 6/reproducibility.md §5
quenais-run --molecule N2 --basis sto-3g \
    --geometry "N 0 0 0; N 0 0 2.1" \
    --solver gqe --gqe-seed 101 \
    --project-dir runs/N2_r2.1_rep4 --steps 0 1 2 3 4
```

**If ScH stage 0 takes hours, something is wrong** — most likely `eigsh` got a
bad starting vector, or threads are not set. Expected: minutes.

### The standing practice

> **Run every calculation twice. Diff the fingerprints, not just the energy.**

A result that has not been run twice and fingerprinted is not a result — it is a
candidate for one of the five failure modes in `docs/reproducibility.md`.